# Lecture 1: integrating the equation of motion, and how accurate it is

Orbital Mechanics, Technical University of Munich, winter semester 2026/27.

The two-body equation of motion, $\ddot{\mathbf r} = -\mu\,\mathbf r/r^3$, has an exact solution (Lectures 2 to 5). That makes it the ideal test for a numerical integrator: we can measure the error exactly. This notebook

1. writes the equation as a first-order system,
2. integrates it with a fixed-step Runge-Kutta method written out in full,
3. shows how the error depends on the step size,
4. uses an adaptive integrator and shows what its tolerances mean,
5. looks at how the error grows over many revolutions.

**Conventions.** Units are km and s. A vector name ends with its basis: `r_N` holds components in the inertial basis N.

Copyright © 2026 Dr. Marcello Romano. MIT License; see the file `LICENSE` of the repository.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import pyastrodyn_edu as ad

mu = ad.MU_EARTH                       # km^3/s^2
print("pyastrodyn_edu version", ad.__version__)

## The test orbit

An Earth orbit with semi-major axis 11000 km and eccentricity 0.35. The exact state at any time comes from Kepler's equation (`propagate_kepler_rv`), which we use as the reference.

In [ ]:
a, e = 11000.0, 0.35
i, raan, argp, f = np.radians([52.0, 30.0, 70.0, 20.0])
r0_N, v0_N = ad.coe2rv(a, e, i, raan, argp, f, mu)
T = ad.period(a, mu)

print("initial position (km)  ", np.round(r0_N, 3))
print("initial velocity (km/s)", np.round(v0_N, 5))
print(f"period {T:.1f} s = {T/60:.1f} min")

def position_error(r_N, t):
    """Distance (km) between a computed position and the exact one at time t."""
    r_exact_N, _ = ad.propagate_kepler_rv(r0_N, v0_N, t, mu)
    return np.linalg.norm(r_N - r_exact_N, axis=-1)

## 1. The equation of motion as a first-order system

An integrator advances a state $\mathbf x$ given its rate of change $\dot{\mathbf x} = \mathbf f(\mathbf x)$. We stack position and velocity, $\mathbf x = [\mathbf r;\ \mathbf v]$, so that $\dot{\mathbf x} = [\mathbf v;\ -\mu\,\mathbf r/r^3]$. That is the function `twobody_eom`.

In [ ]:
x0_N = np.concatenate((r0_N, v0_N))
xdot_N = ad.twobody_eom(0.0, x0_N, mu)

print("state      x  =", np.round(x0_N, 4))
print("derivative x' =", np.round(xdot_N, 6))
print("first three entries of x' equal the velocity:", np.allclose(xdot_N[:3], v0_N))

## 2. A fixed-step integrator: Runge-Kutta 4

`rk4_step` advances the state by one step $h$ using four evaluations of the derivative; `propagate_rk4` repeats it. Open `pyastrodyn_edu/twobody.py` to read both: together they are about ten lines.

One revolution with 100 steps:

In [ ]:
n_steps = 100
t, r_N, v_N = ad.propagate_rk4(r0_N, v0_N, T / n_steps, n_steps, mu)

print(f"step size {T/n_steps:.1f} s")
print(f"position error after one revolution: {position_error(r_N[-1], T):.3f} km")

## 3. The error against the step size

Runge-Kutta 4 is a method of order 4: over a fixed time span the error is proportional to $h^4$. Halving the step should divide the error by about 16. On a logarithmic plot the error against the number of steps is a straight line of slope $-4$, until rounding takes over.

In [ ]:
steps = np.array([50, 100, 200, 400, 800, 1600, 3200, 6400])
errors = []
for n in steps:
    _, r_N, _ = ad.propagate_rk4(r0_N, v0_N, T / n, int(n), mu)
    errors.append(position_error(r_N[-1], T))
errors = np.array(errors)

print(" steps    error (km)    ratio to previous")
for k, (n, err) in enumerate(zip(steps, errors)):
    ratio = f"{errors[k-1]/err:8.1f}" if k else "       -"
    print(f"{n:6d}   {err:11.3e}   {ratio}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(steps, errors, "o-", label="Runge-Kutta 4")
ax.loglog(steps, errors[0] * (steps[0] / steps) ** 4, "--", color="0.5", label="slope $-4$")
ax.set_xlabel("steps per revolution")
ax.set_ylabel("position error after one revolution (km)")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

## 4. An adaptive integrator and its tolerances

`propagate_numerical` uses SciPy's `solve_ivp`. The integrator estimates the error of every step and adapts the step size so that, for each component of the state,

$$\text{error of the step} \le \texttt{atol} + \texttt{rtol}\times|\text{value}|.$$

SciPy's own defaults are `rtol = 1e-3` and `atol = 1e-6`. They are meant for general use and are far too loose for an orbit. The library's defaults are `rtol = 1e-11` and an `atol` tied to the size of the initial state.

In [ ]:
cases = [
    ("SciPy defaults, method RK45",   dict(method="RK45",   rtol=1e-3,  atol=1e-6)),
    ("SciPy defaults, method DOP853", dict(method="DOP853", rtol=1e-3,  atol=1e-6)),
    ("rtol = 1e-6,  method DOP853",   dict(method="DOP853", rtol=1e-6)),
    ("rtol = 1e-9,  method DOP853",   dict(method="DOP853", rtol=1e-9)),
    ("library defaults (rtol 1e-11)", dict()),
]

print(f"{'settings':<34}{'1 revolution':>16}{'10 revolutions':>18}")
for label, options in cases:
    r_N, _ = ad.propagate_numerical(r0_N, v0_N, [T, 10 * T], mu, **options)
    err = position_error(r_N, np.array([T, 10 * T]))
    print(f"{label:<34}{err[0]:13.3e} km{err[1]:15.3e} km")

Two lessons. First, the tolerance is a bound on the error of **each step**, not on the final error: the final error is much larger than `rtol` times the orbit size, and it grows with time. Second, always set the tolerances yourself when you integrate an orbit.

## 5. Many revolutions

The error does not grow linearly. A small error in the energy is an error in the period, so the computed spacecraft slowly runs ahead of, or behind, the true one. The plot shows the position error over 100 revolutions for the adaptive integrator with the library defaults, and for Runge-Kutta 4 with 400 steps per revolution.

In [ ]:
revolutions = np.arange(1, 101)
times = revolutions * T

r_adaptive_N, v_adaptive_N = ad.propagate_numerical(r0_N, v0_N, times, mu)
error_adaptive = position_error(r_adaptive_N, times)

per_revolution = 400
t_rk4, r_rk4_N, v_rk4_N = ad.propagate_rk4(r0_N, v0_N, T / per_revolution, 100 * per_revolution, mu)
r_rk4_N = r_rk4_N[per_revolution::per_revolution]          # keep the state at the end of each revolution
v_rk4_N = v_rk4_N[per_revolution::per_revolution]
error_rk4 = position_error(r_rk4_N, times)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(revolutions, error_rk4, label="Runge-Kutta 4, 400 steps per revolution")
ax.loglog(revolutions, error_adaptive, label="adaptive DOP853, library defaults")
ax.set_xlabel("revolutions")
ax.set_ylabel("position error (km)")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

print(f"after 100 revolutions:  Runge-Kutta 4  {error_rk4[-1]:.3e} km    adaptive  {error_adaptive[-1]:.3e} km")

A check that needs no exact solution: the specific energy must stay constant. Its drift is a direct measure of the quality of an integration, and it is available for any problem, also when no exact solution exists.

In [ ]:
energy0 = ad.specific_energy(r0_N, v0_N, mu)
drift_adaptive = np.abs(ad.specific_energy(r_adaptive_N, v_adaptive_N, mu) / energy0 - 1.0)
drift_rk4 = np.abs(ad.specific_energy(r_rk4_N, v_rk4_N, mu) / energy0 - 1.0)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogy(revolutions, drift_rk4, label="Runge-Kutta 4, 400 steps per revolution")
ax.semilogy(revolutions, drift_adaptive, label="adaptive DOP853, library defaults")
ax.set_xlabel("revolutions")
ax.set_ylabel("relative error of the specific energy")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

## 6. The result does not depend on the unit system

The same integration in canonical units (length unit = the semi-major axis, time unit such that $\mu = 1$). Because the library ties the absolute tolerance to the size of the initial state, the two runs agree to well within the error of either one.

In [ ]:
canonical = ad.UnitSystem.canonical(mu, a)
r0_c = canonical.nondim(r0_N, "length")
v0_c = canonical.nondim(v0_N, "speed")
T_c = canonical.nondim(T, "time")

r_c, _ = ad.propagate_numerical(r0_c, v0_c, [10 * T_c], 1.0)       # mu = 1
r_km, _ = ad.propagate_numerical(r0_N, v0_N, [10 * T], mu)

print(f"period in canonical units: {T_c:.6f}  (2 pi = {2*np.pi:.6f})")
print("difference between the two runs after 10 revolutions:",
      f"{np.linalg.norm(canonical.dim(r_c[0], 'length') - r_km[0]):.2e} km")

### The trap: a fixed absolute tolerance

So, does scaling the variables change the accuracy? The table compares the position error after 10 revolutions in the two unit systems, for four settings.

- With a **fixed step**, nothing changes: scaling the variables and the step together leaves the arithmetic the same.
- With a tolerance that is **relative, or scaled to the state** (the library default), the two runs agree to a few percent.
- With a **fixed absolute tolerance** the same number means different things. `atol = 1e-9` is a micrometre when the unit is the kilometre, but about a centimetre when the unit is the semi-major axis: a tolerance ten thousand times looser.

In [ ]:
n_rev = 10
r_exact_N, _ = ad.propagate_kepler_rv(r0_N, v0_N, n_rev * T, mu)

def error_km(**options):
    r_N, _ = ad.propagate_numerical(r0_N, v0_N, [n_rev * T], mu, **options)
    return np.linalg.norm(r_N[0] - r_exact_N)

def error_canonical(**options):
    r, _ = ad.propagate_numerical(r0_c, v0_c, [n_rev * T_c], 1.0, **options)
    return np.linalg.norm(canonical.dim(r[0], "length") - r_exact_N)

# Fixed step: the same number of steps per revolution in both unit systems.
n = 400
_, r_N, _ = ad.propagate_rk4(r0_N, v0_N, T / n, n_rev * n, mu)
_, r_c, _ = ad.propagate_rk4(r0_c, v0_c, T_c / n, n_rev * n, 1.0)
rk4_km = np.linalg.norm(r_N[-1] - r_exact_N)
rk4_canonical = np.linalg.norm(canonical.dim(r_c[-1], "length") - r_exact_N)

print(f"{'settings':<52}{'km and s':>14}{'canonical':>14}   (position error, km)")
print(f"{'Runge-Kutta 4, 400 steps per revolution':<52}{rk4_km:14.4e}{rk4_canonical:14.4e}")
for label, options in [
    ("adaptive, library defaults (atol scaled to the state)", dict()),
    ("adaptive, rtol 1e-9, atol scaled to the state", dict(rtol=1e-9)),
    ("adaptive, rtol 1e-11, atol fixed at 1e-9", dict(rtol=1e-11, atol=1e-9)),
    ("adaptive, rtol 1e-9, atol fixed at 1e-6", dict(rtol=1e-9, atol=1e-6)),
]:
    print(f"{label:<52}{error_km(**options):14.4e}{error_canonical(**options):14.4e}")

Scaling does not make an integration more accurate by itself: double-precision arithmetic carries the same number of significant digits at 7000 km as at 1. What scaling changes is the **meaning of an absolute tolerance**. In canonical units every component of the state is of order one, so one number is a sensible tolerance for all of them. In km and s the position is about a thousand times larger than the velocity, and one number treats them unequally. The library avoids the question by tying the default absolute tolerance to the initial state.